In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/revision/supplement-extra"
setup_plotting(figure_dir, display_dpi=200, save_dpi=300)

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

In [ ]:
import ast

from matplotlib.colors import TwoSlopeNorm


def parse_sim_codetections(x: str) -> np.ndarray:
    return np.asarray(ast.literal_eval(x), dtype=int)


def p_to_stars(p: float) -> str:
    if not np.isfinite(p):
        return ""
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return ""


def build_symmetric_matrices(
    df: pd.DataFrame,
    gene1_col: str = "gene1",
    gene2_col: str = "gene2",
    z_col: str = "z_score",
    p_adj_col: str = "p_value_adj",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    genes = sorted(set(df[gene1_col]).union(df[gene2_col]))
    zmat = pd.DataFrame(np.nan, index=genes, columns=genes, dtype=float)
    pmat = pd.DataFrame(np.nan, index=genes, columns=genes, dtype=float)

    for r in df[[gene1_col, gene2_col, z_col, p_adj_col]].itertuples(index=False):
        g1, g2, z, p = r
        zmat.loc[g1, g2] = z
        zmat.loc[g2, g1] = z
        pmat.loc[g1, g2] = p
        pmat.loc[g2, g1] = p

    # Set diagonal values directly on DataFrame to avoid read-only array issues
    for gene in genes:
        zmat.loc[gene, gene] = 0.0
        pmat.loc[gene, gene] = np.nan
    return zmat, pmat


def plot_gene_gene_heatmap(
    zmat: pd.DataFrame,
    pmat: pd.DataFrame,
    title: str = "Gene-by-gene z-score (stars: adj. p-value)",
    stars_fontsize: int = 8,
) -> tuple[plt.Figure, plt.Axes]:
    genes = list(zmat.index)
    n = len(genes)

    fig_w = max(6, 0.35 * n + 2)
    fig_h = max(6, 0.35 * n + 2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    finite = np.isfinite(zmat.values)
    vmax = np.nanmax(np.abs(zmat.values[finite])) if finite.any() else 1.0
    norm = TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)

    im = ax.imshow(zmat.values, norm=norm, cmap="coolwarm")
    cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.04)
    cb.set_label("z-score")

    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(genes, rotation=90, fontsize=18)
    ax.set_yticklabels(genes, fontsize=18)
    ax.set_title(title)

    # stars inside cells (based on adjusted p-values)
    for i, gi in enumerate(genes):
        for j, gj in enumerate(genes):
            s = p_to_stars(pmat.loc[gi, gj])
            if s:
                ax.text(j, i, s, ha="center", va="center", fontsize=stars_fontsize)

    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)
    fig.tight_layout()
    return fig, ax

In [ ]:
def plot_permutation_test_result(
    x_sim,
    x_obs,
    p_value,
    xlabel="number of codetections",
    ylabel="count",
    title="",
    text_offset_x=None,
    text_offset_y=None,
    save_path=None,
    figsize=(4, 3),
):
    sns.set_theme(style="ticks", context="paper")
    fig, ax = plt.subplots(figsize=figsize)
    hist = sns.histplot(x_sim, ax=ax)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    bar_centers = [p.get_x() + p.get_width() / 2 for p in hist.patches]
    print(bar_centers)

    # draw red vertical line for observed number of clonal clusters
    ax.axvline(x_obs, color="red", linestyle="--")
    # add text for observed number of clonal clusters
    ymax = ax.get_ylim()[1]
    ax.text(
        0.995 * x_obs if text_offset_x is None else text_offset_x,
        0.88 * ymax if text_offset_y is None else text_offset_y,
        f"Observed {xlabel}: {x_obs}\nempirical p-value: {np.round(p_value, 5)}",
        color="red",
        ha="right",
        fontsize=8,
    )
    ax.tick_params(axis="x", labelsize=8)
    ax.tick_params(axis="y", labelsize=8)

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)

    ax.set_title(title, fontsize=10)

    # split x axis into two parts
    sns.despine(ax=ax)
    if save_path is not None:
        plt.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
            transparent=True,
        )
    plt.show()

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/08.1-kidney_tcr_clonal_clusters.h5ad"
adata = sc.read_h5ad(path)
adata

In [ ]:
[g for g in adata.var_names if "TRB" in g]

## Check codetection of TRAV and TRBV with CD3 genes

In [ ]:
result = pd.read_csv(
    "results/revision/codetection_permutation_result_aggr_genes_trv_cd3.csv"
)
result

In [ ]:
for pair in [
    ("TRAV", "TRBV"),
    ("TRAV", "CD3"),
    ("TRBV", "CD3"),
    ("TRAV", "TRAC"),
    ("TRBV", "TRBC"),
]:
    mask = (result["gene1"] == pair[0]) & (result["gene2"] == pair[1])
    result_pair = result[mask]
    plot_permutation_test_result(
        np.asarray([result_pair["sim_codetections"].iloc[0].strip("[]").split(", ")])
        .astype(int)
        .flatten(),
        result_pair["obs_codetections"].iloc[0],
        result_pair["p_value_adj"].iloc[0],
        # xlabel=f"{pair[0]} codetections with {pair[1]}",
        title=f"{pair[0]} co-occurrences with {pair[1]}",
        xlabel="number of co-occurrences",
        # save_path=f"figures/revision/rebuttal/codetection_{pair[0]}_with_{pair[1]}.pdf",
        save_path=os.path.join(figure_dir, f"codetection_{pair[0]}_with_{pair[1]}.pdf"),
    )

## Check codetection of TRAV genes with each other

In [ ]:
result = pd.read_csv("results/revision/codetection_permutation_result_av_genes.csv")
result["sim_codetections"] = result["sim_codetections"].apply(
    lambda x: np.asarray(x.strip("[]").split(", ")).astype(int)
)

sim = np.array(result["sim_codetections"].tolist())
E_sim = sim.mean(axis=1)
std_sim = sim.std(axis=1)

result["z_score"] = (result["obs_codetections"] - E_sim) / std_sim
result

In [ ]:
zmat, pmat = build_symmetric_matrices(result, z_col="z_score", p_adj_col="p_value_adj")
fig, ax = plot_gene_gene_heatmap(zmat, pmat)
plt.show()

In [ ]:
result = pd.read_csv("results/revision/codetection_permutation_result_bv_genes.csv")
result["sim_codetections"] = result["sim_codetections"].apply(
    lambda x: np.asarray(x.strip("[]").split(", ")).astype(int)
)

sim = np.array(result["sim_codetections"].tolist())
E_sim = sim.mean(axis=1)
std_sim = sim.std(axis=1)

result["z_score"] = (result["obs_codetections"] - E_sim) / std_sim
zmat, pmat = build_symmetric_matrices(result, z_col="z_score", p_adj_col="p_value_adj")
fig, ax = plot_gene_gene_heatmap(zmat, pmat)
plt.show()